### Installation

In [1]:
!pip install -U -q langchain

In [2]:
# Installing the OpenAI integration
!pip install -U -q langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 500.5/500.5 kB 22.0 MB/s eta 0:00:00


In [3]:
!pip install -q langchain-text-splitters langchain-community bs4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [4]:
!pip install -U -q "langchain[openai]"

### Chat Model

In [5]:
import os
from langchain_openai import ChatOpenAI
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

model = ChatOpenAI(model="gpt-4o")

### Embeedings Model

In [6]:
# OpenAI Embeeding Model
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

### Vector Store

In [7]:
!pip install -U -q "langchain-core"

In [8]:
# In Memory Vector Store
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embeddings)

In [9]:
# ChromaDB Vector Store

In [ ]:
# !pip install -qU langchain-chroma

In [ ]:
# from langchain_chroma import Chroma

# vector_store = Chroma(
#     collection_name="example_collection",
#     embedding_function=embeddings,
#     persist_directory="./chroma_langchain_db",  # Where to save data locally, remove if not necessary
# )

### Indexing

##### Loading Data

In [10]:
import bs4
from langchain_community.document_loaders import WebBaseLoader

# Only keep post title, headers, and content from the full HTML.
bs4_strainer = bs4.SoupStrainer(class_=("post-title", "post-header", "post-content"))
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs={"parse_only": bs4_strainer},
)
docs = loader.load()

assert len(docs) == 1
print(f"Total characters: {len(docs[0].page_content)}")

Total characters: 43047


In [11]:
print(docs[0].page_content[:500])



      LLM Powered Autonomous Agents
    
Date: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng


Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.
Agent System Overview#
In


### Splitting documents

In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  # chunk size (characters)
    chunk_overlap=200,  # chunk overlap (characters)
    add_start_index=True,  # track index in original document
)
all_splits = text_splitter.split_documents(docs)

print(f"Split blog post into {len(all_splits)} sub-documents.")

Split blog post into 63 sub-documents.


### Storing Document

In [13]:
document_ids = vector_store.add_documents(documents=all_splits)

print(document_ids[:3])

['878edad8-cf24-4d56-83ba-e74fa4c01320', '59e5c223-d537-4d03-8918-346c81928140', 'e05df8d3-70fe-41fc-8174-5001242baf66']


### RAG

In [ ]:
# Create Retriever
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}  # retrieve top 5 chunks
)

In [24]:
from langchain_core.prompts import PromptTemplate

rag_prompt = PromptTemplate.from_template("""
You are a Retrieval-Augmented Generation assistant.

Use the context below to answer the question.
If the answer is partially available, summarize it clearly.
Only say "I don't have enough information from the documents."
if the context is completely unrelated.

Context:
{context}

Question:
{question}

Answer:
""")

In [25]:
def rag_assistant(question):
    # Retrieve relevant chunks
    docs = retriever.invoke(question)

    # Combine retrieved text
    context = "\n\n".join([doc.page_content for doc in docs])

    if not context.strip():
      return "No relevant documents found."

    # Format prompt
    final_prompt = rag_prompt.format(
        context=context,
        question=question
    )

    # Generate answer
    response = model.invoke(final_prompt)

    return response.content, docs

In [26]:
def chat():
    print("RAG Assistant Ready (type 'exit' to quit)\n")

    while True:
        question = input("You: ")

        if question.lower() == "exit":
            break

        answer, sources = rag_assistant(question)

        print("\nAssistant:", answer)
        print("\nSources Used:")
        for i, doc in enumerate(sources):
            print(f"{i+1}. {doc.metadata.get('source', 'Unknown')}")
        print("\n" + "-"*50 + "\n")

In [27]:
chat()

RAG Assistant Ready (type 'exit' to quit)

You: What are autonomous agents?

Assistant: Autonomous agents, as described in the context, are systems that utilize a large language model (LLM) as their core controller, with additional components like planning, memory, and reflection mechanisms. These agents, such as AutoGPT and generative agents, can break down complex tasks into manageable subgoals, learn from past actions for self-improvement, and simulate human-like behavior in interactive applications. They are designed to function independently, making decisions without user intervention, though they still face challenges related to reliability in their natural language interfaces.

Sources Used:
1. https://lilianweng.github.io/posts/2023-06-23-agent/
2. https://lilianweng.github.io/posts/2023-06-23-agent/
3. https://lilianweng.github.io/posts/2023-06-23-agent/
4. https://lilianweng.github.io/posts/2023-06-23-agent/
5. https://lilianweng.github.io/posts/2023-06-23-agent/

-----------